# Notes about Simons Algorithm 

- first register must be exactly like it was initially. 
- copying of first registers bit value can be copied to 2nd register bit by using cnot gates that will overwrite |0> into | f(x) > 
- say cnot gate between 1st qubit of first register and first qubit of 2nd register. if first qubit is 0.. 2nd qubit will also be zero. if first qubit is one the second will have X applied to the input and changed to 1. 
- running this circuit multiple times gives output for each time as z 
- we get systems of equation s.z = 0 s1.z1 = 0 s2.z2=0 and so on 
- we then solve that systems of equations classically by checking for orthogonality and then infer our secret string from that operation. z is not the secret string. 


In [ ]:
from qiskit import QuantumCircuit, transpile 
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2
import numpy as np
from qiskit.visualization import plot_histogram
import itertools
import pandas as pd
from qiskit.quantum_info import Statevector



In [5]:
def stringDotProduct(str1, str2):
    result = 0
    for i in range(len(str1)):
        if (str1[i] == '1' and str2[i] == '1'):
            result += 1
    return (result%2)

def simonSolver(orthogonal_set):
    # This is basically the same as saying you have less equations than unknowns.
    if len(orthogonal_set) < len(orthogonal_set[0]):
        print("You need more strings!!!!!!!!")
        return ""

    # We define n to be the length of the strings. 
    # We assume all of them to be of the same length.
    n = len(orthogonal_set[0])

    # This basically goes over all possible integer values from 0 all the way
    # up to 2**n-1, the biggest number you can represent with n bits. For each
    # value, it converts it to binary form, padding the beginning with 0s to 
    # make its length n.
    setOfAllPossibleBitStrings = [ format(b, '0'+str(n)+'b') for b in range(2**n) ]

    # Using brute force method
    # the algorithm just goes over all possible bitstrings and checks
    # whether that particular string is orthogonal to each bitstring in the orthogonal_set.
    # If it is, it equates that to the result.
    # One important thing to notice is 000..0 bitstring is always orthonogal to any set
    # of bitstrings, and since setOfAllPossibleBitStrings contains the bitstrings in increasing
    # order, we return the latest one as our result.
    result = ""
    for string in setOfAllPossibleBitStrings:
        orthogonal_counter = 0
        for orthoElement in orthogonal_set:
            if (stringDotProduct(string, orthoElement) == 0):
                orthogonal_counter += 1

        if (orthogonal_counter == len(orthogonal_set)):
            result = string
    return result

# This function is used to generate the oracle for the Simon's algorithm.
def simonOracle(s):
    # Generate a random bit string s with length n
    #s = "".join(['1' if np.random.randint(2) == 1 else '0' for _ in range(n)])
    
    # Initialize the quantum circuit for the oracle
    n = len(s)+1
    oracle_circuit = QuantumCircuit(2*n)
    
    # Apply CNOT gates to copy the first register to the second register
    for i in range(n):
        oracle_circuit.cx(i, n+i)
    
    # Find indices where s[i] = 1
    setOfIndices = [i for i in range(n) if s[i] == "1"]

    # Check if s is not all zeros
    if setOfIndices:
        least_significant_index = setOfIndices[-1]

        # Apply CNOTs controlled by least_significant_index
        for target in setOfIndices:
            oracle_circuit.cx(least_significant_index, n+target)

        # Apply X gates to the second register
        for i in range(n):
            oracle_circuit.x(n+i)

    # Convert the oracle circuit to a gate with a label

    return oracle_circuit


def reverse_dict_keys(d):
    return {key[::-1]: value for key, value in d.items()}


# Make truth table of the oracle. we need this to confirm the secret string
# is fulfilling the promise. 
# Function to generate truth table for the oracle (for verification)
def oracle_truth_table(n, s):
    """Print the truth table of the oracle to verify f(x) = f(x ⊕ s)."""
    print(f"\nTruth Table for Oracle with s = {s}:")
    print("Input (x) | Output (f(x))")
    print("-" * 25)
    
    simulator = AerSimulator()
    sampler = SamplerV2(mode=simulator)
    
    for x_int in range(2**n):
        x = format(x_int, f'0{n}b')
        qc = QuantumCircuit(n)
        
        # Set input state
        for i, bit in enumerate(x[::-1]):  # Qiskit’s little-endian order
            if bit == '1':
                qc.x(i)
        
        # Apply oracle
        qc.append(simonOracle(n,s), range(n))
        
        # Measure output register
        qc.measure_all()
        result = sampler.run([transpile(qc, simulator)], shots=1).result()
        output = list(result[0].data.meas.get_counts().keys())[0]
        f_x = output[:n]  # Output is second n bits (little-endian)
        
        print(f"{x}      | {f_x}")

# must have qc.measure_all() run before calling this function
def runCircuit(qc):
    backend = AerSimulator(method = 'automatic', precision = 'single')
    sampler = SamplerV2(mode=backend)
    qc = transpile(qc, backend=backend)
    job = sampler.run([qc])
    result = job.result()
    qc_counts = result[0].data.meas.get_counts()
    return result, qc_counts


# now checking manually 

def CheckManually():
    simulator = AerSimulator()
    sampler = SamplerV2(mode=simulator)

    for x_int in range(2**n):
        # For all the possibillities depending on n
        qc = QuantumCircuit(2*n)
        qc.append(simonOracle(n))
        x = format(x_int, f'0{n}b')
        print(x) 

        # # Measure output register
        # qc.measure_all()
        # result = sampler.run([transpile(qc, simulator)], shots=1).result()
        # output = list(result[0].data.meas.get_counts().keys())[0]
        # f_x = output[:n]  # Output is second n bits (little-endian)
        # return f_x

# We need a total of 2n quantum bits, n for the input and n for the output.
n = 3
s = "101"
qc = QuantumCircuit(n)

for i in range(n): # Apply Hadamard gates to the first n qubits
    qc.h(i)
qc.barrier()

# Apply the oracle
oracle = simonOracle(s) 

qc.append(oracle_gate, range(2*n))

qc.barrier()
qc.h(range(n))

# now measure all 
qc.measure_all()


result, qc_counts = runCircuit(qc)
plot_histogram(qc_counts)

# The qc counts are not z strings.. z strings are the first registers values
# For each result we get.. least

orthogonal_set = []
for bitstring in qc_counts.keys():
    reversed_bitstring = bitstring[::-1]
    first_n_elements = reversed_bitstring[0:n]
    # we get only first the n qubit values like from 0011 we get 00. 
    # note that this is after reversing. so before it would 1100 
    # after reversing its 0011 and then we get 00 from it. that's equal to z string
    # We do not want repeated values so only getting unique values
    if not( first_n_elements in orthogonal_set):
        orthogonal_set.append( first_n_elements)

print(orthogonal_set)
# These generate a set that secret string s is orthogonal to. 


# all quantum part done. 
# run the result through simonsolver that return the s that is orthogonal to all elements 
# in the orthogonal_set list. 

secret_string_result = simonSolver(orthogonal_set)
print("Secret string we found is: ", secret_string_result)
oracle_truth_table(2, "101")



# First checking the function. 


IndexError: string index out of range

In [4]:
# import random
import qiskit.quantum_info as qi
from qiskit import QuantumCircuit
import numpy as np
from qiskit_aer import AerSimulator
from qiskit import ClassicalRegister
import numpy as np
import galois

def simon_function(s: str):
    """
    Create a QuantumCircuit implementing a query gate for Simon problem obeying the promise for the hidden string `s`
    """
    # Our quantum circuit has 2n qubits for n = len(s)
    n = len(s)
    qc = QuantumCircuit(2 * n)

    # Define a random permutation of all n bit strings. This permutation will effectively hide the string s.
    pi = np.random.permutation(2**n)
    print(pi)
    # Now we'll define a query gate explicitly. The idea is to first define a function g(x) = min{x,x ^ s}, which
    # is a simple function that satisfies the promise, and then we take f to be the composition of g and the random
    # permutation pi. This gives us a random function satisfying the promise for s.

    query_gate = np.zeros((4**n, 4**n))
    for x in range(2**n):
        for y in range(2**n):
            z = y ^ pi[min(x, x ^ int(s, 2))]
            query_gate[x + 2**n * z, x + 2**n * y] = 1

    # Our circuit has just this one query gate
    qc.unitary(query_gate, range(2 * n))
    return qc



def simon_measurements(problem: QuantumCircuit, k: int):
    """
    Quantum part of Simon's algorithm. Given a `QuantumCircuit` that
    implements f, get `k` measurements to be post-processed later.
    """
    n = problem.num_qubits // 2

    qc = QuantumCircuit(2 * n, n)
    qc.h(range(n))
    qc.compose(problem, inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))

    result = AerSimulator().run(qc, shots=k, memory=True).result()
    return result.get_memory()


def simon_algorithm(problem: QuantumCircuit):
    """
    Given a `QuantumCircuit` that implements a query gate for Simon problem, return the hidden string `s`.
    """

    # Quantum part: run the circuit defined previously k times and gather the measurement results.
    # Replace +10 by +r for any nonnegative integer r depending on desired confidence.

    measurements = simon_measurements(problem, k=problem.num_qubits // 2 + 10)
    print("Measurement results:")
    display(measurements)

    # Classical post-processing:

    # 1. Convert measurements of form '11101' to 2D-array of integers
    matrix = np.array([list(bitstring) for bitstring in measurements]).astype(int)

    # 2. Interpret matrix as using arithmetic mod 2, and find null space
    null_space = galois.GF(2)(matrix).null_space()
    print("Null space:")
    display(null_space)

    # 3. Convert back to a string
    print("Guess for hidden string s:")
    if len(null_space) == 0:
        # No non-trivial solution; `s` is all-zeros
        return "0" * len(measurements[0])
    return "".join(np.array(null_space[0]).astype(str))


display(simon_algorithm(simon_function("10011")))

[ 3 13  0  9 17 28 23 20  2  1  8 19  5  6 25 21 31 27 11 10 15 16 26  7
 14 29 30 22 18  4 12 24]
Measurement results:


['01000',
 '11110',
 '01000',
 '01100',
 '11101',
 '00011',
 '01111',
 '00111',
 '11101',
 '00011',
 '10110',
 '01011',
 '00100',
 '01011',
 '11101']

Null space:


GF([[1, 0, 0, 1, 1]], order=2)

Guess for hidden string s:


'10011'